# LJP Criminal Dataset - Reason 컬럼 구조화

이 노트북은 `lbox/lbox_open` 데이터셋의 `ljp_criminal` 서브셋에서 `reason` 컬럼을 구조화된 JSON으로 변환합니다.

## 구조화 목표

- **패턴 타입 감지**: 5가지 패턴 자동 감지
- **양형 이유 추출**: 불리한/유리한 정상, 법률상 처단형 범위, 양형기준 등
- **신상정보 추출**: 등록의무, 공개명령 면제, 취업제한 등
- **JSON 출력**: 구조화된 형식으로 저장

## 1. 환경 설정

In [ ]:
!pip install datasets huggingface-hub -q

In [ ]:
from datasets import load_dataset
import json
import re
from typing import Dict, List, Optional, Any
from enum import Enum
from collections import Counter

## 2. 데이터셋 로드

In [ ]:
# ljp_criminal 데이터셋 로드
dataset = load_dataset("lbox/lbox_open", "ljp_criminal", split="train")

print(f"총 샘플 수: {len(dataset)}")
print(f"컬럼: {dataset.column_names}")

## 3. 구조화 함수 정의

In [ ]:
# structure_reason.py 파일의 내용을 여기에 복사하거나 import
# 간단하게 하기 위해 주요 함수만 정의

class ReasonPatternType(Enum):
    """Reason 패턴 타입"""
    BASIC_NARRATIVE = "기본 서술형"
    STRUCTURED_GUIDELINE = "구조화된 양형기준형"
    OMITTED = "약식 생략형"
    WITH_PERSONAL_INFO = "신상정보 포함형"
    HYBRID = "혼합형"


def detect_pattern_type(text: str) -> ReasonPatternType:
    """패턴 타입 감지"""
    if re.search(r'양형의\s*이유', text) and re.search(r'^\s*생략\s*$', text, re.MULTILINE):
        return ReasonPatternType.OMITTED
    
    has_statutory = bool(re.search(r'법률상\s+처단형의\s+범위', text))
    has_guideline = bool(re.search(r'양형기준에\s+따른\s+권고형의\s+범위', text))
    has_decision = bool(re.search(r'선고형의\s+결정', text))
    
    if has_statutory and has_guideline and has_decision:
        return ReasonPatternType.STRUCTURED_GUIDELINE
    
    has_personal_info = bool(re.search(r'신상정보\s+(?:등록|제출의무)', text))
    has_disclosure = bool(re.search(r'(?:공개|고지)명령', text))
    has_numbering = bool(re.search(r'^\s*\d+\.\s+', text, re.MULTILINE))
    
    if has_numbering and not (has_statutory and has_guideline):
        if has_personal_info or has_disclosure:
            return ReasonPatternType.WITH_PERSONAL_INFO
        return ReasonPatternType.HYBRID
    
    if has_personal_info or has_disclosure:
        return ReasonPatternType.WITH_PERSONAL_INFO
    
    return ReasonPatternType.BASIC_NARRATIVE


# 이 셀을 실행하면 structure_reason.py의 전체 내용이 로드됩니다
# 또는 파일을 직접 import할 수도 있습니다
# from structure_reason import structure_reason, detect_pattern_type

print("구조화 함수 로드 완료!")

## 4. 패턴 분포 분석

In [ ]:
# 전체 데이터셋의 패턴 분포 확인
pattern_distribution = Counter()

for example in dataset:
    pattern = detect_pattern_type(example['reason'])
    pattern_distribution[pattern.value] += 1

print("패턴 타입 분포:")
print("="*50)
for pattern, count in pattern_distribution.most_common():
    percentage = (count / len(dataset)) * 100
    print(f"{pattern}: {count}개 ({percentage:.1f}%)")

# 시각화
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
patterns = list(pattern_distribution.keys())
counts = list(pattern_distribution.values())
plt.bar(patterns, counts)
plt.xlabel('패턴 타입')
plt.ylabel('개수')
plt.title('Reason 컬럼 패턴 분포')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 5. 단일 샘플 구조화 예시

In [ ]:
# structure_reason.py를 import (파일이 같은 디렉토리에 있어야 함)
from structure_reason import structure_reason

# 샘플 83번 구조화
sample_idx = 83
reason_text = dataset[sample_idx]['reason']
case_id = dataset[sample_idx]['id']

structured = structure_reason(reason_text, case_id)

# raw_text 제외하고 출력
output = structured.copy()
del output['raw_text']

print(json.dumps(output, ensure_ascii=False, indent=2))

## 6. 전체 데이터셋 구조화

In [ ]:
from structure_reason import structure_dataset

print("전체 데이터셋 구조화 시작...")
print("(시간이 걸릴 수 있습니다)")

# 전체 데이터셋 구조화
structured_dataset = structure_dataset(dataset)

print(f"\n구조화 완료!")
print(f"새로운 컬럼: {structured_dataset.column_names}")

## 7. 구조화 결과 확인

In [ ]:
# 각 패턴 타입별로 샘플 하나씩 출력
shown_patterns = set()

for example in structured_dataset:
    pattern = example['reason_pattern_type']
    
    if pattern not in shown_patterns:
        shown_patterns.add(pattern)
        
        print(f"\n{'='*80}")
        print(f"패턴 타입: {pattern}")
        print(f"케이스 ID: {example['id']}")
        print('='*80)
        
        # structured 데이터에서 raw_text 제외하고 출력
        structured = example['reason_structured']
        output = {
            'case_id': structured['case_id'],
            'pattern_type': structured['pattern_type'],
            'sentencing_reason': structured['sentencing_reason'],
            'personal_information': structured['personal_information']
        }
        
        print(json.dumps(output, ensure_ascii=False, indent=2))
        
        if len(shown_patterns) >= 5:
            break

## 8. 통계 분석

In [ ]:
# 구조화 통계
print("구조화 통계")
print("="*80)

# 불리한 정상이 있는 케이스
with_unfavorable = sum(
    1 for ex in structured_dataset 
    if ex['reason_structured']['sentencing_reason'].get('unfavorable_factors')
)

# 유리한 정상이 있는 케이스
with_favorable = sum(
    1 for ex in structured_dataset 
    if ex['reason_structured']['sentencing_reason'].get('favorable_factors')
)

# 신상정보 등록이 필요한 케이스
with_registration = sum(
    1 for ex in structured_dataset 
    if ex['reason_structured']['personal_information'].get('registration_required')
)

# 공개명령이 면제된 케이스
with_disclosure_exempt = sum(
    1 for ex in structured_dataset 
    if ex['reason_structured']['personal_information'].get('disclosure_exempted')
)

print(f"총 케이스 수: {len(structured_dataset)}")
print(f"\n불리한 정상 추출: {with_unfavorable}개 ({with_unfavorable/len(structured_dataset)*100:.1f}%)")
print(f"유리한 정상 추출: {with_favorable}개 ({with_favorable/len(structured_dataset)*100:.1f}%)")
print(f"\n신상정보 등록 필요: {with_registration}개 ({with_registration/len(structured_dataset)*100:.1f}%)")
print(f"공개명령 면제: {with_disclosure_exempt}개 ({with_disclosure_exempt/len(structured_dataset)*100:.1f}%)")

## 9. JSON 파일로 저장

In [ ]:
# 전체 데이터를 JSON Lines 형식으로 저장
output_file = 'ljp_criminal_reason_structured.jsonl'

with open(output_file, 'w', encoding='utf-8') as f:
    for example in structured_dataset:
        # 저장할 데이터 준비
        output_data = {
            'id': example['id'],
            'casetype': example['casetype'],
            'casename': example['casename'],
            'label': example['label'],
            'reason_structured': {
                'case_id': example['reason_structured']['case_id'],
                'pattern_type': example['reason_structured']['pattern_type'],
                'sentencing_reason': example['reason_structured']['sentencing_reason'],
                'personal_information': example['reason_structured']['personal_information']
            }
        }
        
        f.write(json.dumps(output_data, ensure_ascii=False) + '\n')

print(f"✅ 저장 완료: {output_file}")
print(f"총 {len(structured_dataset)}개 케이스가 저장되었습니다.")

In [ ]:
# 단일 JSON 파일로도 저장 (용량이 클 수 있음)
# output_file_json = 'ljp_criminal_reason_structured.json'

# all_data = []
# for example in structured_dataset:
#     output_data = {
#         'id': example['id'],
#         'casetype': example['casetype'],
#         'casename': example['casename'],
#         'label': example['label'],
#         'reason_structured': {
#             'case_id': example['reason_structured']['case_id'],
#             'pattern_type': example['reason_structured']['pattern_type'],
#             'sentencing_reason': example['reason_structured']['sentencing_reason'],
#             'personal_information': example['reason_structured']['personal_information']
#         }
#     }
#     all_data.append(output_data)

# with open(output_file_json, 'w', encoding='utf-8') as f:
#     json.dump(all_data, f, ensure_ascii=False, indent=2)

# print(f"✅ 저장 완료: {output_file_json}")

## 10. 저장된 파일 확인

In [ ]:
# 저장된 JSONL 파일 읽어서 확인
with open(output_file, 'r', encoding='utf-8') as f:
    first_line = f.readline()
    first_data = json.loads(first_line)

print("저장된 데이터 형식 (첫 번째 케이스):")
print("="*80)
print(json.dumps(first_data, ensure_ascii=False, indent=2))

## 요약

✅ **완료된 작업**:

1. **패턴 감지**: 5가지 패턴 타입 자동 분류
2. **구조화**: 양형 이유 및 신상정보를 구조화된 JSON으로 변환
3. **추출 항목**:
   - 법률상 처단형의 범위
   - 양형기준 (유형, 특별양형인자, 권고형 범위)
   - 불리한 정상 / 유리한 정상
   - 선고형 결정 이유
   - 신상정보 등록/제출의무
   - 공개명령/고지명령 면제
   - 취업제한
4. **저장**: JSONL 형식으로 파일 저장

✅ **활용 가능성**:

- 법률 판결 예측 모델 학습
- 양형 요소 분석
- 유사 판례 검색
- 법률 문서 요약
- 양형 패턴 연구